# Stock Market Data Analysis and Dashboard
**Tesla (TSLA) and GameStop (GME)**

This notebook completes the six requested tasks: retrieve stock data with `yfinance`, extract revenue tables from the web where available, display the requested first/last five rows, and plot stock prices and quarterly revenue.

> Run the cells from top to bottom in Google Colab. Online sources may change their page structure or block automated requests. If a source blocks scraping, the notebook reports the issue and attempts a Yahoo Finance fallback for quarterly revenue.


## Setup

In [ ]:
!pip -q install yfinance pandas requests beautifulsoup4 lxml html5lib plotly


In [ ]:
import re
import warnings
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


## Helper functions

In [ ]:
def clean_revenue_table(table):
    """Normalize a two-column date/revenue table and keep numeric revenue rows."""
    if table is None or table.empty:
        return pd.DataFrame(columns=["Date", "Revenue"])

    table = table.copy()
    table.columns = [str(c).strip() for c in table.columns]
    # Find likely date and revenue columns by header text; otherwise use first two columns.
    date_col = next((c for c in table.columns if "date" in c.lower() or "quarter" in c.lower()), table.columns[0])
    revenue_col = next((c for c in table.columns if "revenue" in c.lower()), table.columns[1] if len(table.columns) > 1 else table.columns[0])
    table = table[[date_col, revenue_col]].copy()
    table.columns = ["Date", "Revenue"]
    table["Date"] = pd.to_datetime(table["Date"], errors="coerce")
    table["Revenue"] = (
        table["Revenue"].astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
    )
    table["Revenue"] = pd.to_numeric(table["Revenue"], errors="coerce")
    table = table.dropna(subset=["Date", "Revenue"]).sort_values("Date").reset_index(drop=True)
    return table


def scrape_revenue(url, company_name):
    """Try to scrape a revenue table. Macrotrends may block automated access."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; educational notebook)"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    tables = pd.read_html(response.text)
    candidates = []
    for table in tables:
        if len(table.columns) >= 2:
            cleaned = clean_revenue_table(table)
            if not cleaned.empty:
                candidates.append(cleaned)
    if not candidates:
        raise ValueError(f"No usable revenue table found for {company_name}.")
    # Prefer a table with a substantial number of rows.
    return max(candidates, key=len)


def yahoo_revenue_fallback(ticker_symbol):
    """Fallback to Yahoo Finance quarterly financials if web scraping is blocked."""
    ticker = yf.Ticker(ticker_symbol)
    stmt = ticker.quarterly_income_stmt
    if stmt is None or stmt.empty:
        stmt = ticker.quarterly_financials
    if stmt is None or stmt.empty:
        raise ValueError(f"No quarterly financial statement available for {ticker_symbol}.")
    possible = [idx for idx in stmt.index if "total revenue" in str(idx).lower() or str(idx).lower() == "revenue"]
    if not possible:
        raise ValueError(f"Revenue row not found in Yahoo Finance statement for {ticker_symbol}.")
    row = stmt.loc[possible[0]]
    data = pd.DataFrame({"Date": pd.to_datetime(row.index, errors="coerce"), "Revenue": pd.to_numeric(row.values, errors="coerce")})
    return data.dropna().sort_values("Date").reset_index(drop=True)


## Question 1 — Extract Tesla stock data

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

# Reset the index and display the first five rows as requested.
tesla_data.reset_index(inplace=True)
tesla_data.head()


## Question 2 — Extract Tesla revenue using web scraping

In [ ]:
tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"

try:
    tesla_revenue = scrape_revenue(tesla_revenue_url, "Tesla")
    tesla_revenue_source = "Web scraping (Macrotrends)"
except Exception as error:
    print("Web scraping was blocked or the page format changed:", error)
    print("Using Yahoo Finance quarterly revenue as a fallback.")
    tesla_revenue = yahoo_revenue_fallback("TSLA")
    tesla_revenue_source = "Yahoo Finance fallback"

print("Revenue source:", tesla_revenue_source)
tesla_revenue.tail()


## Question 3 — Extract GameStop stock data

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")

# Reset the index and display the first five rows as requested.
gme_data.reset_index(inplace=True)
gme_data.head()


## Question 4 — Extract GameStop revenue using web scraping

In [ ]:
gme_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"

try:
    gme_revenue = scrape_revenue(gme_revenue_url, "GameStop")
    gme_revenue_source = "Web scraping (Macrotrends)"
except Exception as error:
    print("Web scraping was blocked or the page format changed:", error)
    print("Using Yahoo Finance quarterly revenue as a fallback.")
    gme_revenue = yahoo_revenue_fallback("GME")
    gme_revenue_source = "Yahoo Finance fallback"

print("Revenue source:", gme_revenue_source)
gme_revenue.tail()


## Plotting function

In [ ]:
def make_graph(stock_data, revenue_data, title):
    stock = stock_data.copy()
    revenue = revenue_data.copy()

    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    revenue["Date"] = pd.to_datetime(revenue["Date"], errors="coerce")
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")
    stock = stock.dropna(subset=["Date", "Close"])
    revenue = revenue.dropna(subset=["Date", "Revenue"])

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("Historical Share Price", "Quarterly Revenue"),
        vertical_spacing=0.15
    )
    fig.add_trace(
        go.Scatter(x=stock["Date"], y=stock["Close"], mode="lines", name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=revenue["Date"], y=revenue["Revenue"], name="Revenue"),
        row=2, col=1
    )
    fig.update_yaxes(title_text="Share Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD; source units)", row=2, col=1)
    fig.update_layout(title=title, height=750, template="plotly_white")
    fig.show()
    return fig


## Question 5 — Plot Tesla stock graph

In [ ]:
tesla_fig = make_graph(
    tesla_data,
    tesla_revenue,
    "Tesla Stock Price and Quarterly Revenue"
)


## Question 6 — Plot GameStop stock graph

In [ ]:
gme_fig = make_graph(
    gme_data,
    gme_revenue,
    "GameStop Stock Price and Quarterly Revenue"
)


## Save evidence screenshots
Run this final cell after the tables and charts have been displayed. It saves the requested tables as PNG files. The two interactive Plotly charts can be saved separately from their chart toolbar (camera icon) as PNG images.

If Kaleido is available, the cell also attempts to save the charts as PNG files. If not, use the camera icon on each chart or take a screenshot with `Win + Shift + S`.


In [ ]:
from pathlib import Path

evidence_dir = Path("evidence")
evidence_dir.mkdir(exist_ok=True)

# Export tables as HTML so they can be opened and captured clearly if needed.
tesla_data.head().to_html(evidence_dir / "Q1_Tesla_Stock.html", index=False)
tesla_revenue.tail().to_html(evidence_dir / "Q2_Tesla_Revenue.html", index=False)
gme_data.head().to_html(evidence_dir / "Q3_GameStop_Stock.html", index=False)
gme_revenue.tail().to_html(evidence_dir / "Q4_GameStop_Revenue.html", index=False)

try:
    tesla_fig.write_image(str(evidence_dir / "Q5_Tesla_Graph.png"))
    gme_fig.write_image(str(evidence_dir / "Q6_GameStop_Graph.png"))
    print("Saved table HTML files and both chart PNG files in:", evidence_dir.resolve())
except Exception as error:
    print("Tables saved as HTML. Plotly PNG export needs Kaleido; use each chart's camera icon instead.")
    print("Details:", error)
    print("Evidence folder:", evidence_dir.resolve())


## Submission checklist
- [ ] Question 1: screenshot of `tesla_data.head()` with the reset-index code.
- [ ] Question 2: screenshot of `tesla_revenue.tail()`.
- [ ] Question 3: screenshot of `gme_data.head()` with the reset-index code.
- [ ] Question 4: screenshot of `gme_revenue.tail()`.
- [ ] Question 5: screenshot of Tesla graph.
- [ ] Question 6: screenshot of GameStop graph.
- [ ] Download this notebook as `.ipynb` and upload it to GitHub.
